# NLP Preprocessing & Representation Cheat Sheet

---

# 1. Tokenization

## What
Splits raw text into tokens (words, subwords, characters).

---

## Types to Know

### Word Tokenization
Splits on whitespace/punctuation.

Example:
```python
"I love NLP!" → ["I", "love", "NLP"]
```

### Subword Tokenization
- BERT → WordPiece

- GPT → BPE (Byte Pair Encoding)

Example:
```python
"playing" → ["play", "##ing"]
```

`##` means:
> "I am attached to the previous token"

### Character Tokenization
```python
"cat" → ["c", "a", "t"]
```

---

## Key Interview Points

- BERT uses **WordPiece**
- GPT uses **BPE**
- BPE merges most frequent character pairs bottom-up
- WordPiece merges pairs maximizing language model likelihood
- Tokenization directly affects embeddings and model quality

---

## Silent Failures

### Languages Without Spaces
Chinese/Japanese fail with naive whitespace tokenizers.

### Domain Terms
```python
"COVID-19" → ["COVID", "-", "19"]
```

### Hinglish / OOV
Unknown tokens become:
```python
[UNK]
```
Meaning gets lost entirely.

---

# 2. Stemming

## What
Rule-based suffix stripping to get root form.

Output is NOT necessarily a real word.

---

## Algorithms

### Porter Stemmer
Most common, English only.

### Snowball Stemmer
Improved Porter, multilingual.

### Lancaster Stemmer
Most aggressive.

---

## Examples

```python
"happily" → "happili"
```

---

## Key Interview Points

- Fast
- No vocabulary needed
- Language agnostic
- Produces stems, not real words
- Cannot handle irregular forms

Example:
```python
"ran" ≠ "run"
```

---

## Silent Failures

### Overstemming
```python
"university"
"universe"
"universal"

→ "univers"
```

False semantic merge.

### Understemming
```python
"ran" and "run"
```
remain different.

### Embedding Failure
Broken stems don't exist in embeddings:
```python
"happili"
```

---

# 3. Lemmatization

## What
Returns actual dictionary base form (lemma).

Uses:
- vocabulary
- morphological analysis
- POS tagging

---

## Examples

```python
"better" → "good"
"was" → "be"
```

---

## Key Interview Points

- Needs POS tags
- Returns valid dictionary words
- Slower than stemming
- Embedding-safe
- Transformers usually don't require it

Example:
```python
"meeting"

Noun → "meeting"
Verb → "meet"
```

---

## Silent Failures

### Wrong POS Tag
Wrong upstream POS → wrong lemma.

### OOV Words
```python
"tweeted"
"googled"
```
may fail in WordNet-based systems.

### Multi-word Expression Damage
```python
"project management" → "project manage"
```

---

## One-Liner

> Stemming is fast and crude. Lemmatization is slow and linguistically correct.

---

# 4. Named Entity Recognition (NER)

## What
Identifies and classifies named entities.

Examples:
- PERSON
- ORG
- GPE
- DATE
- MONEY
- PRODUCT

---

## How It Works

### Classical
CRF + BIO tagging.

### Modern
Transformer/BERT fine-tuning.

---

## BIO Tagging

| Tag | Meaning |
|---|---|
| B | Beginning |
| I | Inside |
| O | Outside |

Example:
```python
Barack Obama lives in Washington

Barack   → B-PER
Obama    → I-PER
lives    → O
Washington → B-GPE
```

---

## Key Interview Points

- Originated from DARPA MUC conferences
- Foundation for:
  - Relation extraction
  - Knowledge graphs
  - QA systems

---

## Silent Failures

### Ambiguity
```python
"Apple"
```
fruit or company?

### Emerging Entities
Old models may miss:
```python
"ChatGPT"
```

### Coreference Failure
```python
"He founded Tesla"
```
"He" not resolved.

### Indian Entity Failure
Western-trained models miss:
- SEBI
- NSE
- Nifty 50

### Nested Entities
```python
"Bank of India building"
```

---

## Entity Linking

Maps mention → knowledge base node.

Example:
```python
"Taj Mahal" → Agra, India
```

---

# 5. Stop Words

## What
High-frequency low-information words removed.

Examples:
```python
the, is, in, and
```

---

## Key Interview Points

- Reduces feature space
- Speeds up training
- NLTK has 179 English stop words
- Always whitelist negations

Keep:
```python
not, never, no, don't
```

- Transformers usually don't require stop-word removal

---

## Silent Failures

### Negation Destruction
```python
"not good" → "good"
```

### Legal/Financial Catastrophe
```python
"not liable" → "liable"
```

### Named Entity Fragmentation
```python
"Bank of India" → "Bank India"
```

### Empty Sentences
```python
"To be or not to be" → []
```

---




# 6. One Hot Encoding (OHE)

## What
Each word gets a sparse vector.

Vocabulary size = vector dimension.

---

## Example

```python
Vocabulary = ["cat", "dog", "fish"]

cat  → [1,0,0]
dog  → [0,1,0]
fish → [0,0,1]
```

---

## Key Interview Points

- Sparse vectors
- Useful for low-cardinality categorical data
- No semantic meaning captured

---

## Silent Failures

### Synonym Blindness
```python
"laptop" and "notebook computer"
```
completely unrelated.

### Antonym Blindness
```python
"good" and "bad"
```
equally distant.

### Curse of Dimensionality
50k vocabulary → 50k-dimensional vectors.

### OOV Problem
Unseen words impossible to represent.

---

## One-Liner

> Use OHE only for small fixed categorical features.

---

# 7. Bag of Words (BoW)

## What
Represents text using word frequencies.

Vector size = vocabulary size.

---

## Variants

### Binary BoW
```python
1 → present
0 → absent
```

### Count BoW
Raw frequency counts.

### TF-IDF
Weights words by:
- frequency in document
- rarity across corpus

Preferred in production.

---

## Key Interview Points

```python
"dog bites man"
"man bites dog"
```

→ identical vectors.

Word order is lost.

- TF-IDF reduces document length bias
- Still widely used in:
  - Spam detection
  - Search engines
  - Elasticsearch

---

## Silent Failures

### Negation
```python
"not good"
```
may become:
```python
"good"
```

### Sarcasm
```python
"Oh great, another delay"
```

### Polysemy
```python
"python"
```
snake vs programming language.

### Short Text
Tweets produce sparse vectors.

---

# 8. N-Gram Bag of Words

## What
Uses consecutive word sequences as features.

---

## Types

### Unigram
```python
["good", "movie"]
```

### Bigram
```python
["not good", "battery life"]
```

### Trigram
```python
["you have won"]
```

---

## Key Interview Points

- Best practical setup:
  - Unigrams + bigrams + TF-IDF
- Vocabulary grows exponentially

Formula:
```python
V^N
```

Example:
```python
10,000 words

Bigrams  → 100M
Trigrams → 1T
```

- Beyond trigrams usually hurts more than helps

---

## Silent Failures

### Vocabulary Explosion
Huge sparse feature space.

### Long-Range Dependencies
```python
"food ... was absolutely not worth it"
```

Negation too far away.

### Cross-Sentence Noise
```python
"great. Battery"
```

creates meaningless bigram.

### Negation Weight Imbalance
```python
"not bad"
```
may still be dominated by `"bad"`.

Fix:
```python
"not_bad"
```

---

# Master Cheat Sheet

| Concept | One-Liner |
|---|---|
| Tokenization | Bad split breaks everything downstream |
| Stemming | Fast, crude, breaks embeddings |
| Lemmatization | Slow, linguistically correct |
| NER | Grounds text to real-world entities |
| Stop Words | Always whitelist negations |
| OHE | Only for low-cardinality categories |
| BoW TF-IDF | Fast, interpretable, production-valid |
| N-Gram BoW | Unigrams + bigrams = sweet spot |

---

# Three Cross-Cutting Concepts

## 1. Error Cascading
One component's mistake silently propagates downstream.

---

## 2. Distribution Mismatch
Model trained on one population fails on another.

---

## 3. OOV Problem
Unseen words become invisible features.

Meaning gets silently destroyed.

---

### The full classical pipeline flow

1) Raw text comes in
2) Lowercase it
3) Whitespace/punctuation tokenization — split into word tokens
4) Stopword removal — drop "the," "a," "is," "of," etc. to cut the vocabulary and reduce noise dimensions
5) Stemming or lemmatization — collapse word variants to roots
6) Feed remaining tokens into BoW (just count occurrences) or TF-IDF (count weighted by how rare the word is across all documents)
7) You now have a sparse matrix: rows = documents, columns = vocab terms, values = counts or TF-IDF weights
8) Feed that sparse matrix into SVM, Naive Bayes, logistic regression, etc.